In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# Load the dataset
df = pd.read_csv("data/Employee.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

In [ ]:
from pathlib import Path
import joblib

# Find the project root from the notebook's current working directory
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

backend_dir = project_root / "backend"
backend_dir.mkdir(parents=True, exist_ok=True)

model_path = backend_dir / "attrition_model.joblib"

joblib.dump(best_model, model_path)

print("Model saved successfully!")
print("Location:", model_path.resolve())
print("File exists:", model_path.exists())

In [ ]:
# Column names
print("Columns:")
print(df.columns.tolist())

# Data types and non-null counts
df.info()

# Summary statistics
display(df.describe(include="all").T)

In [ ]:
print("Missing values per column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nTarget values:")
print(df["LeaveOrNot"].value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, column in zip(
    axes,
    ["City", "Gender", "Education"]
):
    rates = (
        df.groupby(column, observed=True)["LeaveOrNot"]
        .mean()
        .mul(100)
        .reset_index(name="Attrition Rate (%)")
    )

    sns.barplot(
        data=rates,
        x=column,
        y="Attrition Rate (%)",
        ax=ax
    )

    ax.set_title(f"Attrition Rate by {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Attrition Rate (%)")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
required_columns = [
    "Education",
    "JoiningYear",
    "City",
    "PaymentTier",
    "Age",
    "Gender",
    "EverBenched",
    "ExperienceInCurrentDomain",
    "LeaveOrNot"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing required columns:", missing_columns)
else:
    print("All required columns are present.")

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Check the required columns
required_columns = [
    "Education",
    "JoiningYear",
    "City",
    "PaymentTier",
    "Age",
    "Gender",
    "EverBenched",
    "ExperienceInCurrentDomain",
    "LeaveOrNot",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")
print("\nTarget values:")
print(df["LeaveOrNot"].value_counts(dropna=False))

print("\nMissing values:")
print(df[required_columns].isnull().sum())

In [ ]:
# Keep only the columns specified in the assignment
data = df[required_columns].copy()

# Remove records without a known target
data = data.dropna(subset=["LeaveOrNot"])

# Convert target values to numeric
data["LeaveOrNot"] = pd.to_numeric(
    data["LeaveOrNot"],
    errors="coerce"
)

# Remove invalid target values
data = data.dropna(subset=["LeaveOrNot"])

# Ensure the target is binary: 0 = Stay, 1 = Leave
data = data[data["LeaveOrNot"].isin([0, 1])].copy()
data["LeaveOrNot"] = data["LeaveOrNot"].astype(int)

# Remove exact duplicate records
data = data.drop_duplicates().reset_index(drop=True)

print("Original rows:", len(df))
print("Rows after cleaning:", len(data))
print("Rows removed:", len(df) - len(data))
print("\nMissing values after target cleaning:")
print(data.isnull().sum())

In [ ]:
X = data.drop(columns=["LeaveOrNot"])
y = data["LeaveOrNot"]

categorical_features = [
    "Education",
    "City",
    "Gender",
    "EverBenched",
]

numerical_features = [
    "JoiningYear",
    "PaymentTier",
    "Age",
    "ExperienceInCurrentDomain",
]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

In [ ]:
from sklearn.model_selection import train_test_split

# First, set aside 15% for final testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y,
)

# Split the remaining 85% into training and validation
# 15 / 85 = approximately 17.65%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=y_train_val,
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Process categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# Process numerical features
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Apply the correct processing to each column
preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numerical_pipeline, numerical_features),
])

print("Preprocessing pipeline created.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

trained_models = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    print(f"{name} trained successfully.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

results = []

for name, model in trained_models.items():
    predictions = model.predict(X_val)
    probabilities = model.predict_proba(X_val)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_val, probabilities),
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(
    axes,
    trained_models.items()
):
    predictions = model.predict(X_val)

    sns.heatmap(
        confusion_matrix(y_val, predictions),
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")

plt.tight_layout()
plt.show()

In [ ]:
display(results_df.round(4))

In [ ]:
best_model_name = results_df["F1-score"].idxmax()
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)
print("\nValidation results:")
print(results_df.loc[best_model_name].round(4))

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    accuracy_score
)
import matplotlib.pyplot as plt

test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)[:, 1]

print("Selected model:", best_model_name)
print("\nTest classification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Stay", "Leave"],
    zero_division=0
))

print("Test ROC-AUC:",
      round(roc_auc_score(y_test, test_probabilities), 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["Stay", "Leave"],
    cmap="Blues"
)

plt.title(f"Test Confusion Matrix — {best_model_name}")
plt.show()

In [ ]:
from pathlib import Path
import joblib

backend_dir = Path("../backend")
backend_dir.mkdir(parents=True, exist_ok=True)

model_path = backend_dir / "attrition_model.joblib"

joblib.dump(best_model, model_path)

print("Model saved successfully!")
print("Saved to:", model_path.resolve())